On structure l'EDA en sections, de la plus générale à la plus précise. Pour chaque section, je te donne les questions à te poser et les outils à utiliser, pas les réponses.

## Avant de commencer

- **Travaille sur `train.csv`** : c'est le seul fichier qui contient `Survived`. `test.csv` sert à Kaggle, pas à ton analyse.
- **Ne modifie rien pendant l'EDA** : tu observes et tu notes. Le nettoyage vient après.
- **Première cellule** : tous les imports, la constante `RANDOM_STATE = 42` et le chemin du CSV en relatif.

## Le plan - EDA

| Section | Question à te poser | Outils |
|---|---|---|
| 1. Premier regard | Combien de lignes et de colonnes ? Quels types ? | `.shape`, `.info()`, `.head()`, `.dtypes` |
| 2. La cible | Les deux classes de `Survived` sont-elles équilibrées ? Qu'est-ce que ça change pour le choix de la métrique plus tard ? | `.value_counts(normalize=True)`, countplot |
| 3. Valeurs manquantes | Quelles colonnes en ont, et combien ? Est-ce que le manque est aléatoire ou lié à autre chose (la classe, par exemple) ? | `.isna().mean()`, heatmap des manquants |
| 4. Variables une par une | Pour les numériques : quelle forme, quelles valeurs extrêmes ? Pour les catégorielles : combien de modalités, lesquelles sont rares ? | `.describe()`, histplot, boxplot, `.value_counts()` |
| 5. Chaque variable face à la survie | Le taux de survie change-t-il selon la variable ? | `groupby(...)["Survived"].mean()`, barplot, `pd.crosstab` |
| 6. Croisements | Deux variables ensemble racontent-elles autre chose qu'une seule ? | catplot avec `hue`, tableau croisé à deux entrées |
| 7. Corrélations | Quelles variables numériques bougent ensemble ? | `.corr()`, heatmap |
| 8. Valeurs aberrantes | Ce sont des erreurs ou des cas réels ? Faut-il y toucher ? | boxplot, méthode de l'IQR |
| 9. Hypothèses | Ce que tu vas vérifier, puis utiliser au nettoyage et au feature engineering | Cellule markdown |

## Les bonnes pratiques

- **Après chaque graphique, une cellule markdown** avec ce que tu observes en une ou deux phrases. C'est ce que le jury évalue, bien plus que le graphique lui-même.
- **Un graphique répété plusieurs fois devient une fonction** (par exemple taux de survie par variable). C'est plus propre et plus lisible.
- **Graphiques titrés, axes nommés en français** : ils te serviront pour les slides du musée. Enregistre les meilleurs dans `reports/figures/`.
- **Ouvre une liste « à traiter au nettoyage »** au fil de l'EDA : manquants, colonnes à supprimer, texte à exploiter.
- **Corrélation n'est pas causalité** : garde ça en tête pour la question du musée, « quels facteurs ont *réellement* influencé ».

Commence par les sections 1 à 3, puis montre-moi tes observations si tu veux un retour.

In [28]:
# LIBRAIRIES
import pandas as pd

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from pathlib import Path


# DIRECTORIES
DATA_DIR = Path("..") / "data"
RAW_DATA_PATH = DATA_DIR / "raw" / "train.csv"
FIGURES_DIR = Path("..") / "reports" / "figures"

# OPTIONS
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

In [29]:
# LE DATASET
df = pd.read_csv(RAW_DATA_PATH)

In [30]:
df

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
...,...,...,...,...,...,...,...,...,...,...,...,...
886,887,0,2,"Montvila, Rev. Juozas",male,27.0,0,0,211536,13.0000,NaN,S
887,888,1,1,"Graham, Miss. Margaret Edith",female,19.0,0,0,112053,30.0000,B42,S
888,889,0,3,"Johnston, Miss. Catherine Helen ""Carrie""",female,NaN,1,2,W./C. 6607,23.4500,NaN,S
889,890,1,1,"Behr, Mr. Karl Howell",male,26.0,0,0,111369,30.0000,C148,C


Data Dictionary

|Variable|Definition|Key|
|---|---|---|
|survival|Survival|0 = No, 1 = Yes|
|pclass|Ticket|class 1 = 1st, 2 = 2nd, 3 = 3rd|
|sex|Sex||
|Age|Age in years||
|sibsp| # of siblings / spouses aboard the Titanic|
|parch| # of parents / children aboard the Titanic|	
|ticket| Ticket number|
|fare|Passenger fare|
|cabin|Cabin number|
|embarked|Port of Embarkation|	C = Cherbourg, Q = Queenstown, S = Southampton|
    
Variable Notes
pclass: A proxy for socio-economic status (SES)
1st = Upper
2nd = Middle
3rd = Lower

age: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

sibsp: The dataset defines family relations in this way...
Sibling = brother, sister, stepbrother, stepsister
Spouse = husband, wife (mistresses and fiancés were ignored)

parch: The dataset defines family relations in this way...
Parent = mother, father
Child = daughter, son, stepdaughter, stepson
Some children travelled only with a nanny, therefore parch=0 for them.

In [31]:
df.shape

(891, 12)

À conclure : combien de passagers et combien de colonnes ? Est-ce que chaque ligne correspond bien à un passager ?
- 891 Passagers avec 12 colonnes

In [45]:
df.duplicated("Name").sum()

np.int64(0)

In [32]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 83.7 KB


In [47]:
summary = pd.DataFrame({
    "type": df.dtypes,
    "nb_uniques": df.nunique(),
    "pct_uniques": (df.nunique() / len(df) * 100).round(1),
    "nb_manquants": df.isna().sum(),
    "pct_manquants": (df.isna().mean() * 100).round(1),
    "exemples": [df[col].dropna().unique()[:5] for col in df.columns],
})

summary.sort_values("nb_uniques", ascending=False)

,type,nb_uniques,pct_uniques,nb_manquants,pct_manquants,exemples
PassengerId,int64,891,100.0,0,0.0,"[1, 2, 3, 4, 5]"
Name,str,891,100.0,0,0.0,"[Braund, Mr. Owen Harris, Cumings, Mrs. John B..."
Ticket,str,681,76.4,0,0.0,"[A/5 21171, PC 17599, STON/O2. 3101282, 113803..."
Fare,float64,248,27.8,0,0.0,"[7.25, 71.2833, 7.925, 53.1, 8.05]"
Cabin,str,147,16.5,687,77.1,"[C85, C123, E46, G6, C103]"
Age,float64,88,9.9,177,19.9,"[22.0, 38.0, 26.0, 35.0, 54.0]"
SibSp,int64,7,0.8,0,0.0,"[1, 0, 3, 4, 2]"
Parch,int64,7,0.8,0,0.0,"[0, 1, 2, 5, 3]"
Embarked,str,3,0.3,2,0.2,"[S, C, Q]"
Pclass,int64,3,0.3,0,0.0,"[3, 1, 2]"


In [34]:
# LA CIBLE
df["Survived"].value_counts(normalize=True)

Survived
0    0.616162
1    0.383838
Name: proportion, dtype: float64

In [49]:
df.describe(exclude="number")

,Name,Sex,Ticket,Cabin,Embarked
count,891,891,891,204,889
unique,891,2,681,147,3
top,"Braund, Mr. Owen Harris",male,347082,G6,S
freq,1,577,7,4,644


## Premier regard

- **891 passagers, 12 colonnes.** `PassengerId` et `Name` sont uniques à 100 % et aucun nom n'est en double : une ligne correspond bien à un passager.

## Le rôle de chaque colonne

| Colonne | Rôle | Pourquoi |
|---|---|---|
| `PassengerId` | Identifiant | Unique pour chaque ligne, n'apporte rien pour prédire : à supprimer |
| `Survived` | Cible | Binaire, 0 ou 1 |
| `Pclass` | Catégorielle **ordonnée** | Stockée en nombre, mais 1, 2 et 3 sont des rangs, pas des quantités |
| `Sex` | Catégorielle | 2 modalités |
| `Embarked` | Catégorielle | 3 modalités (S, C, Q) |
| `SibSp`, `Parch` | Numériques discrètes | Des comptages de proches à bord, la base d'une variable « taille de la famille » |
| `Age`, `Fare` | Numériques continues | Beaucoup de valeurs différentes |
| `Name` | Texte à exploiter | Unique, donc inutile tel quel, mais contient le titre (Mr, Mrs, Miss, Master…) |
| `Ticket` | Texte à exploiter | Environ 680 valeurs pour 891 passagers, et un billet revient jusqu'à 7 fois : **des groupes voyageaient sur le même billet** |
| `Cabin` | Texte à exploiter ou à supprimer | Plus des trois quarts manquent, et une cabine revient jusqu'à 4 fois (cabines partagées) |

## Valeurs manquantes (premier constat)

| Colonne | Part manquante | Première piste |
|---|---|---|
| `Cabin` | Environ les trois quarts | Trop de manquants pour imputer. Le fait de *ne pas avoir* de cabine renseignée est peut-être lui-même une information |
| `Age` | Environ 1 sur 5 | À imputer, reste à savoir comment |
| `Embarked` | 2 lignes | Très peu, à traiter simplement |

## La cible

- Environ **6 passagers sur 10 sont morts**, 4 sur 10 ont survécu. Le déséquilibre est modéré, pas extrême.
- **Conséquence 1** : il faudra un split **stratifié** (`stratify=y`) pour garder cette proportion dans le train et le test.
- **Conséquence 2** : un modèle qui prédit « mort » pour tout le monde a déjà environ 62 % d'accuracy. **C'est ton score de référence**, tes modèles doivent faire mieux. L'accuracy seule ne suffira pas : il faudra aussi regarder le F1-score ou le rappel.

## Variables texte (`describe`)

- **Environ 2 passagers sur 3 sont des hommes.**
- **Southampton (S) domine** : un peu plus de 7 embarquements sur 10.

## Ce qu'il manque à regarder

| À faire | Pourquoi |
|---|---|
| `df.describe()` sur les numériques | Âge minimum et maximum, écart entre moyenne et médiane de `Fare`, tarifs à 0 (**à vérifier**) |
| `df.duplicated().sum()` | Doublons complets, pas seulement sur le nom |
| Les âges à virgule | D'après la page *Data* de Kaggle, un âge décimal correspond à un bébé de moins d'un an ou à un âge estimé (**à vérifier** sur la page et dans les données) |
| `Fare` dans un groupe | Le tarif est-il par personne ou par billet ? Compare le tarif des passagers qui partagent le même `Ticket` (**hypothèse à vérifier**) |
| Les manquants selon `Pclass` et `Survived` | Pour savoir si le manque est aléatoire ou non. C'est la prochaine section |

Ajoute `describe()` et les doublons complets, puis on attaque les valeurs manquantes.